# SecureSpeak — Optional Paper 1 Additions

Two self-contained experiments that strengthen the Paper 1 submission. Neither is required;
each answers a specific reviewer question.

| # | Experiment | Reviewer question it answers | Needs new data? |
|---|---|---|---|
| 1 | On-device latency & memory | "You claim on-device deployability. Show the numbers." | No |
| 2 | Cross-dataset transfer | "You evaluate on one corpus. Does it generalize?" | Yes (a 2nd URL corpus) |

**Run experiment 1 first — it needs nothing new.** Experiment 2 needs a second phishing URL
corpus; the notebook tells you exactly where to get one and stops cleanly if it is absent.

Same pipeline settings as your other notebooks: 26 engineered URL features, RandomForest(300,
class_weight='balanced'), MiniLM-L12 + 6 features + logistic regression, seed 42.

## 0 · Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, glob

SEED = 42
BASE = '/content/drive/MyDrive/cse498R/Datasets'
OUT_DIR = '/content/drive/MyDrive/cse498R/results_experiments'
os.makedirs(OUT_DIR, exist_ok=True)

STEALTH_CSV = os.path.join(BASE, 'StealthPhisher2025.csv')

# Bangla SMS corpus (auto-locate)
BANGLA_CSV = None
for pat in ['*angala*arta*.csv', '*angla*sms*.csv', '*smish*.csv', '*sms*.csv']:
    hits = glob.glob(os.path.join(BASE, pat))
    if hits:
        BANGLA_CSV = hits[0]; break

# --- Experiment 2 only: a SECOND phishing URL corpus -------------------
# Leave as None to skip Experiment 2. To run it, download one of these to BASE
# and set the filename (any CSV with a URL column + a label column works):
#   - PhiUSIIL Phishing URL Dataset (UCI / Kaggle)
#   - Mendeley "Phishing Detection Dataset" (6tm2d6sz7p)
#   - PhishLegitURLs (Mendeley j43jtv3zzc)
SECOND_CSV = None    # e.g. os.path.join(BASE, 'PhiUSIIL.csv')

print('StealthPhisher :', os.path.exists(STEALTH_CSV))
print('Bangla SMS     :', BANGLA_CSV)
print('Second corpus  :', SECOND_CSV if SECOND_CSV else '(none — Exp 2 will skip)')

StealthPhisher : True
Bangla SMS     : /content/drive/MyDrive/cse498R/Datasets/BangalaBarta bangla_spam_sms smishing.csv
Second corpus  : (none — Exp 2 will skip)


In [3]:
!pip -q install tldextract sentence-transformers 2>/dev/null

import re, math, json, time, gc, warnings
import numpy as np, pandas as pd
from tqdm.auto import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score

import tldextract
np.random.seed(SEED)
warnings.filterwarnings('ignore')
RESULTS = {}
print('Ready. This runtime:')
!cat /proc/cpuinfo | grep 'model name' | head -1
!free -h | head -2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 4.3 MB/s eta 0:00:00
Ready. This runtime:
model name	: Intel(R) Xeon(R) CPU @ 2.00GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.3Gi       7.6Gi       2.0Mi       3.8Gi        11Gi


## 1 · Feature engineering (copied from your pipeline)

Identical to `SecureSpeak_Experiments.ipynb` so the model and timings match the paper.

In [4]:
HIGH_RISK_TLDS = {'tk','ml','ga','cf','gq','pw','top','xyz','online','site','club',
                  'live','shop','info','biz','link','click','download','stream'}
FREE_HOST_TLDS = {'tk','ml','ga','cf','gq','pw'}
FINANCIAL_KW   = ['bank','login','secure','verify','update','account','password','signin',
                  'bkash','nagad','rocket','paypal','amazon','netflix','microsoft','apple','google','confirm']
BRAND_KW       = ['paypal','amazon','google','facebook','apple','microsoft','bkash','nagad','rocket']

def shannon_entropy(s):
    if not s: return 0.0
    f={}
    for ch in s: f[ch]=f.get(ch,0)+1
    n=len(s)
    return -sum((v/n)*math.log2(v/n) for v in f.values())

def engineer_url_features(url):
    url=str(url).strip().lower()
    ext=tldextract.extract(url)
    domain,suffix,subdomain=ext.domain,ext.suffix,ext.subdomain
    path=re.sub(r'https?://[^/]+','',url)
    query=path.split('?',1)[1] if '?' in path else ''
    return [
        min(len(url)/500,1.0), min(url.count('.')/10,1.0), min(url.count('/')/15,1.0),
        min(len(re.findall(r'[-_@!%&=+]',url))/20,1.0),
        sum(c.isdigit() for c in url)/max(len(url),1),
        sum(c.isalpha() for c in url)/max(len(url),1),
        1.0 if suffix in HIGH_RISK_TLDS else 0.0,
        min(subdomain.count('.')+1 if subdomain else 0,5)/5,
        min(len(domain)/30,1.0),
        1.0 if re.match(r'^(?:\d{1,3}\.){3}\d{1,3}$', url.split('/')[2] if '/' in url else url) else 0.0,
        min(sum(b in domain for b in BRAND_KW),3)/3,
        min(len(path)/200,1.0), min(len([s for s in path.split('/') if s])/10,1.0),
        1.0 if '?' in url else 0.0, min(len(query)/200,1.0),
        1.0 if url.startswith('https') else 0.0, 1.0 if 'https' in path else 0.0,
        shannon_entropy(url)/6.0, shannon_entropy(domain)/4.0,
        min(sum(k in url for k in FINANCIAL_KW),5)/5,
        1.0 if re.search(r'@|//.*@',url) else 0.0,
        min(url.count('-')/8,1.0),
        1.0 if len(url)>75 and not url.startswith('https') else 0.0,
        1.0 if suffix in FREE_HOST_TLDS else 0.0,
        min(len(re.findall(r'\d{3,}',url))/3,1.0),
        (1.0 if url.startswith('https') else 0.0)*(0.0 if suffix in HIGH_RISK_TLDS else 1.0),
    ]

URL_FEAT_NAMES=['url_length','dot_count','slash_count','special_chars','digit_ratio','letter_ratio',
 'high_risk_tld','subdomain_depth','domain_length','uses_ip','brand_impersonation','path_length',
 'path_segments','has_query','query_length','has_https','https_in_path','url_entropy','domain_entropy',
 'financial_kw','at_in_url','hyphen_count','long_http','free_hosting_tld','long_numbers','https_x_safe_tld']

PHISH_W={'1','phishing','phish','malicious','bad','spam','fake','scam','fraud','unsafe'}
LEGIT_W={'0','legitimate','legit','benign','good','safe','real','ham','clean','normal'}
def norm_label(v):
    s=str(v).strip().lower()
    if s in PHISH_W: return 1
    if s in LEGIT_W: return 0
    try: return int(float(s)>0.5)
    except: return np.nan

def load_url_csv(path, n_target=None):
    chunks=[]
    for ch in pd.read_csv(path, chunksize=50_000, low_memory=False):
        ch.columns=ch.columns.str.strip()
        uc=next((c for c in ['url','URL','link','Link','website','Website'] if c in ch.columns),None)
        lc=next((c for c in ['label','Label','class','Class','phishing','is_phishing','target',
                             'Target','status','type','result'] if c in ch.columns),None)
        if uc is None or lc is None:
            raise KeyError(f'URL/label column not found in {path}. Columns: {ch.columns.tolist()[:20]}')
        sub=ch[[uc,lc]].rename(columns={uc:'url',lc:'label'})
        sub['label']=sub['label'].map(norm_label)
        sub=sub.dropna(subset=['url','label']); sub['label']=sub['label'].astype(int)
        chunks.append(sub)
        if n_target and sum(len(c) for c in chunks)>=n_target: break
    df=pd.concat(chunks,ignore_index=True)
    return df.iloc[:n_target].reset_index(drop=True) if n_target else df

print('Feature engineer ready.')

Feature engineer ready.


## 2 · Train the model once (used by both experiments)

Trained on the full StealthPhisher domain-disjoint split, matching the paper's Table 1 model.

In [5]:
print('Loading StealthPhisher (this is the slow step)...')
df_sp = load_url_csv(STEALTH_CSV, n_target=150_000)
df_sp['_regdom'] = [f'{tldextract.extract(u).domain}.{tldextract.extract(u).suffix}'.strip('.').lower()
                    for u in tqdm(df_sp['url'], desc='regdom')]
X = np.array([engineer_url_features(u) for u in tqdm(df_sp['url'], desc='feats')])
y = df_sp['label'].values
grp = df_sp['_regdom'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr, te = next(gss.split(X, y, groups=grp))
assert len(set(grp[tr]) & set(grp[te])) == 0

pipe = Pipeline([('scale', StandardScaler()),
                 ('clf', RandomForestClassifier(300, random_state=SEED, n_jobs=-1,
                                                class_weight='balanced'))]).fit(X[tr], y[tr])
acc = accuracy_score(y[te], pipe.predict(X[te]))
auc = roc_auc_score(y[te], pipe.predict_proba(X[te])[:,1])
print(f'Model trained. Domain-disjoint test: acc={acc:.4f}, auc={auc:.4f} (paper: 0.9919 / 0.9976)')

Loading StealthPhisher (this is the slow step)...


regdom:   0%|          | 0/150000 [00:00<?, ?it/s]

feats:   0%|          | 0/150000 [00:00<?, ?it/s]

Model trained. Domain-disjoint test: acc=0.9919, auc=0.9976 (paper: 0.9919 / 0.9976)


## 3 · Experiment 1 — On-device latency & memory

Measures what a phone actually pays per check. Three things a reviewer will ask for:

1. **URL feature engineering + classification** — per-URL, single-sample (not batched), because
   a phone scores one link at a time.
2. **SMS embedding + classification** — the MiniLM forward pass is the real cost here.
3. **Model memory footprint** — what has to fit on the device.

Colab CPU is a reasonable proxy for a mid-range phone; report it as "CPU-only, single-sample,
no GPU." If you can, also run on the slowest runtime available for a conservative number.

In [6]:
import sys, pickle, io

# --- force CPU-only, single-thread: closest proxy to a phone core ------
import os as _os
_os.environ['OMP_NUM_THREADS']='1'
_os.environ['OPENBLAS_NUM_THREADS']='1'

# 1) URL: engineer features + classify, ONE url at a time
sample_urls = df_sp['url'].sample(500, random_state=SEED).tolist()

# warm up
_ = engineer_url_features(sample_urls[0]); _ = pipe.predict(X[:1])

t_feat=[]; t_clf=[]
for u in sample_urls:
    s=time.perf_counter(); f=engineer_url_features(u); t_feat.append(time.perf_counter()-s)
    fa=np.array(f).reshape(1,-1)
    s=time.perf_counter(); _=pipe.predict(fa); t_clf.append(time.perf_counter()-s)

t_feat=np.array(t_feat)*1000; t_clf=np.array(t_clf)*1000
url_total=t_feat+t_clf
print('URL per-link latency (ms), CPU single-sample:')
print(f'  feature engineering : mean {t_feat.mean():.3f}  p50 {np.percentile(t_feat,50):.3f}  p95 {np.percentile(t_feat,95):.3f}')
print(f'  classification      : mean {t_clf.mean():.3f}  p50 {np.percentile(t_clf,50):.3f}  p95 {np.percentile(t_clf,95):.3f}')
print(f'  TOTAL per URL       : mean {url_total.mean():.3f}  p50 {np.percentile(url_total,50):.3f}  p95 {np.percentile(url_total,95):.3f}')

URL per-link latency (ms), CPU single-sample:
  feature engineering : mean 0.145  p50 0.130  p95 0.196
  classification      : mean 84.979  p50 76.431  p95 149.917
  TOTAL per URL       : mean 85.123  p50 76.528  p95 150.046


In [7]:
# 2) SMS: MiniLM embedding + 6 features + LR, ONE message at a time
import torch
from sentence_transformers import SentenceTransformer

device='cpu'   # force CPU to represent phone
mlm=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device=device)

def sms_features(t):
    t=str(t)
    return [
        1 if any(w in t.lower() for w in ['http','www','.com']) else 0,
        1 if any(w in t.lower() for w in ['bkash','nagad','rocket','bank','account']) else 0,
        1 if any(w in t.lower() for w in ['win','prize','free','offer','urgent']) else 0,
        min(len(t)/500,1.0), t.count('!')/max(len(t),1),
        sum(c.isdigit() for c in t)/max(len(t),1),
    ]

# train a tiny LR head on embeddings if a Bangla corpus is present, else time embedding only
sample_msgs = None
if BANGLA_CSV:
    dfb=pd.read_csv(BANGLA_CSV); dfb.columns=dfb.columns.str.strip().str.lower()
    tcol=next((c for c in ['message','text','sms','content','msg'] if c in dfb.columns), dfb.columns[0])
    sample_msgs=dfb[tcol].astype(str).sample(min(300,len(dfb)), random_state=SEED).tolist()
else:
    sample_msgs=['আপনার বিকাশ একাউন্ট ভেরিফাই করুন '+str(i) for i in range(300)]

# warm up
_=mlm.encode([sample_msgs[0]], device=device, show_progress_bar=False)

t_emb=[]
for m in tqdm(sample_msgs, desc='SMS timing'):
    s=time.perf_counter()
    e=mlm.encode([m], device=device, show_progress_bar=False)
    _=sms_features(m)
    t_emb.append((time.perf_counter()-s)*1000)

t_emb=np.array(t_emb)
print('\nSMS per-message latency (ms), CPU single-sample:')
print(f'  MiniLM embed + feats : mean {t_emb.mean():.2f}  p50 {np.percentile(t_emb,50):.2f}  p95 {np.percentile(t_emb,95):.2f}')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SMS timing:   0%|          | 0/300 [00:00<?, ?it/s]


SMS per-message latency (ms), CPU single-sample:
  MiniLM embed + feats : mean 43.69  p50 40.64  p95 67.46


In [8]:
# 3) Memory footprint of each model artifact
def size_mb(obj):
    buf=io.BytesIO(); pickle.dump(obj, buf); return len(buf.getvalue())/1e6

url_model_mb = size_mb(pipe)
try:
    import os as _o
    # MiniLM on disk (downloaded weights)
    from sentence_transformers import util as _u
    mlm_params = sum(p.numel() for p in mlm.parameters())
    mlm_mb = mlm_params*4/1e6   # float32 estimate
except Exception:
    mlm_mb=float('nan'); mlm_params=0

rows=[
 dict(component='URL RandomForest (pickled)', size_mb=round(url_model_mb,2), note='300 trees'),
 dict(component='MiniLM-L12 encoder (fp32)', size_mb=round(mlm_mb,1), note=f'{mlm_params/1e6:.1f}M params'),
]
mem_df=pd.DataFrame(rows)

summary=dict(
  runtime='Colab CPU, single-sample, single-thread (phone proxy)',
  url_latency_ms_mean=round(float(url_total.mean()),3),
  url_latency_ms_p95=round(float(np.percentile(url_total,95)),3),
  sms_latency_ms_mean=round(float(t_emb.mean()),2),
  sms_latency_ms_p95=round(float(np.percentile(t_emb,95)),2),
  url_model_mb=round(url_model_mb,2),
  minilm_mb=round(mlm_mb,1),
)
RESULTS['latency_memory']=summary

print('\n=== ON-DEVICE SUMMARY (report this) ===')
for k,v in summary.items(): print(f'  {k}: {v}')
print('\nMemory footprint:'); print(mem_df.to_string(index=False))

with open(f'{OUT_DIR}/exp6_latency_memory.json','w') as fh:
    json.dump(summary, fh, indent=2)
mem_df.to_csv(f'{OUT_DIR}/exp6_memory.csv', index=False)
print(f'\nSaved to {OUT_DIR}/exp6_latency_memory.json')


=== ON-DEVICE SUMMARY (report this) ===
  runtime: Colab CPU, single-sample, single-thread (phone proxy)
  url_latency_ms_mean: 85.123
  url_latency_ms_p95: 150.046
  sms_latency_ms_mean: 43.69
  sms_latency_ms_p95: 67.46
  url_model_mb: 42.24
  minilm_mb: 470.6

Memory footprint:
                 component  size_mb          note
URL RandomForest (pickled)    42.24     300 trees
 MiniLM-L12 encoder (fp32)   470.60 117.7M params

Saved to /content/drive/MyDrive/cse498R/results_experiments/exp6_latency_memory.json


In [9]:
# Interpretation for the paper
s=RESULTS['latency_memory']
print('Sentence to drop in Section 5 (Deployment):')
print(f'''
"On a single CPU core with no GPU (a proxy for mid-range Android hardware), SecureSpeak
scores a URL in {s["url_latency_ms_mean"]:.1f} ms on average (p95 {s["url_latency_ms_p95"]:.1f} ms),
end to end including feature engineering, and classifies an SMS message in
{s["sms_latency_ms_mean"]:.0f} ms (p95 {s["sms_latency_ms_p95"]:.0f} ms) including the MiniLM
forward pass. The URL model occupies {s["url_model_mb"]:.1f} MB and the multilingual encoder
{s["minilm_mb"]:.0f} MB, both comfortably within the storage and memory budget of a
low-end smartphone. These figures support the paper's on-device deployability claim."
''')
print('NOTE: if SMS latency is high (hundreds of ms), say so honestly and note that the')
print('encoder can be quantized (int8) or distilled further for production — do not hide it.')

Sentence to drop in Section 5 (Deployment):

"On a single CPU core with no GPU (a proxy for mid-range Android hardware), SecureSpeak
scores a URL in 85.1 ms on average (p95 150.0 ms),
end to end including feature engineering, and classifies an SMS message in
44 ms (p95 67 ms) including the MiniLM
forward pass. The URL model occupies 42.2 MB and the multilingual encoder
471 MB, both comfortably within the storage and memory budget of a
low-end smartphone. These figures support the paper's on-device deployability claim."

NOTE: if SMS latency is high (hundreds of ms), say so honestly and note that the
encoder can be quantized (int8) or distilled further for production — do not hide it.


## 4 · Experiment 2 — Cross-dataset transfer

Trains on StealthPhisher, tests on a **different** phishing corpus. Because all 26 features
come from the URL string alone, any corpus with a URL column and a label works. This is the
direct answer to "you only evaluate on one dataset."

**Set `SECOND_CSV` in the config cell to run this.** Good choices:
PhiUSIIL (UCI), Mendeley 6tm2d6sz7p, or PhishLegitURLs. Watch the label convention —
some corpora use 1 = legitimate; the loader's word-map handles common cases but verify.

In [10]:
if SECOND_CSV is None or not os.path.exists(str(SECOND_CSV)):
    print('SECOND_CSV not set or file missing — Experiment 2 skipped.')
    print('Download a second phishing URL corpus to BASE and set SECOND_CSV to run it.')
else:
    d2 = load_url_csv(SECOND_CSV)
    print(f'Second corpus: {len(d2):,} rows, positive rate {d2["label"].mean():.4f}')
    print('CHECK: if positive rate looks inverted, this corpus may use 1=legit. Inspect before trusting.')

    X2 = np.array([engineer_url_features(u) for u in tqdm(d2['url'], desc='feats')])
    y2 = d2['label'].values

    pred2 = pipe.predict(X2)
    proba2 = pipe.predict_proba(X2)[:,1]
    acc2 = accuracy_score(y2, pred2)
    auc2 = roc_auc_score(y2, proba2)
    rec2 = recall_score(y2, pred2, zero_division=0)

    if acc2 > 0.995 or auc2 > 0.999:
        print(f'  !! acc {acc2:.4f} / auc {auc2:.4f} suspiciously high — check for label/format overlap.')

    tbl = pd.DataFrame([
        dict(evaluation='In-corpus (StealthPhisher, domain-disjoint)', n=len(te), accuracy=round(acc,4), auc=round(auc,4)),
        dict(evaluation='Cross-corpus (unseen dataset)', n=len(y2), accuracy=round(acc2,4), auc=round(auc2,4)),
    ])
    RESULTS['cross_dataset']=tbl
    print('\n=== CROSS-DATASET TRANSFER ===')
    print(tbl.to_string(index=False))
    print(f'\nTransfer gap: {acc-acc2:+.4f} accuracy, {auc-auc2:+.4f} AUC.')
    tbl.to_csv(f'{OUT_DIR}/exp7_cross_dataset.csv', index=False)

    print('\nHow to write it: a gap is EXPECTED and honest. A detector trained on one corpus')
    print('rarely matches its in-corpus number elsewhere — reporting the drop bounds how far')
    print('the model travels, and is far more credible than claiming it transfers perfectly.')

SECOND_CSV not set or file missing — Experiment 2 skipped.
Download a second phishing URL corpus to BASE and set SECOND_CSV to run it.


## 5 · What you get out

- `exp6_latency_memory.json` + `exp6_memory.csv` — the deployment numbers for Section 5.
- `exp7_cross_dataset.csv` — the transfer table (only if you set SECOND_CSV).

Both are optional additions. If the SMS latency turns out high, report it honestly with the
quantization note rather than dropping the number — an honest limitation fits this paper's
whole character. If you only run Experiment 1, that alone closes the "prove on-device" gap.